# Semana 04: Automação de Pipelines com GitHub Actions

## Módulo de Automação de CI/CD — Fábrica Virtual Smart N1

Este notebook apresenta o **GitHub Actions** em profundidade, detalhando a construção de workflows em sintaxe **YAML**, a configuração de gatilhos baseados nas regras do **GitFlow**, a execução em *runners* virtuais e a gestão segura de **Secrets**.

### Objetivos de aprendizagem
- Compreender a anatomia completa de um arquivo de workflow `.github/workflows/*.yml`.
- Mapear os eventos do GitFlow (`push` em `develop`, `pull_request` para `main`, tags de release) para gatilhos do GitHub Actions.
- Gerenciar variáveis de ambiente e **GitHub Actions Secrets** (credenciais e tokens de acesso).
- Utilizar a reutilização de tarefas através de *Actions* oficiais e da comunidade.
- Desenvolver e validar workflows YAML utilizando scripts Python.

---


## 1. Fundamentação Teórica

### 1.1 Mapeando Triggers do GitFlow no GitHub Actions

Cada evento na árvore do GitFlow pode acionar workflows específicos no GitHub Actions:

```yaml
name: Pipeline CI/CD GitFlow Smart N1

on:
  # 1. Triggers de Integração e Teste (Feature & Develop)
  pull_request:
    branches: [ develop, main ]
  push:
    branches: [ develop ]

  # 2. Trigger de Publicação Oficial (Tags de Release)
  push:
    tags:
      - 'v*.*.*'
```

---

### 1.2 Estrutura do Workflow YAML Integrado

```yaml
name: Esteira de Validacao CI

on:
  push:
    branches: [ develop, main ]
  pull_request:
    branches: [ develop ]

jobs:
  validar-codigo:
    runs-on: ubuntu-latest
    steps:
      - name: Baixar Código do Repositório
        uses: actions/checkout@v4

      - name: Configurar ambiente Python 3.11
        uses: actions/setup-python@v5
        with:
          python-version: '3.11'

      - name: Instalar Dependências
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt

      - name: Validar Sintaxe e Linting
        run: |
          python -m py_compile app.py
```

---


## 2. Prática — Validador e Parser de Workflows YAML em Python

Nesta atividade prática, desenvolveremos um script Python que lê a estrutura de um arquivo YAML de workflow do GitHub Actions e verifica se as dependências entre *jobs* e *triggers* do GitFlow estão corretamente configuradas.

In [ ]:
workflow_yaml_simulado = """
name: Pipeline CI Smart N1
on:
  push:
    branches:
      - develop
      - main
jobs:
  teste-unitario:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - run: echo "Executando testes..."
  build-imagem:
    needs: teste-unitario
    runs-on: ubuntu-latest
    steps:
      - run: echo "Building..."
"""

def analisar_workflow_github_actions(yaml_content):
    tem_checkout = "actions/checkout" in yaml_content
    tem_gitflow_branches = "develop" in yaml_content and "main" in yaml_content
    tem_dependencias_jobs = "needs:" in yaml_content
    
    return {
        "checkout_configurado": tem_checkout,
        "triggers_gitflow_presentes": tem_gitflow_branches,
        "sequenciamento_jobs_ok": tem_dependencias_jobs,
        "status": "APROVADO" if (tem_checkout and tem_gitflow_branches) else "REPROVADO"
    }

resultado_analise = analisar_workflow_github_actions(workflow_yaml_simulado)
print("=== ANÁLISE DE ESTRUTURA DE PIPELINE GITHUB ACTIONS ===\n")
for k, v in resultado_analise.items():
    print(f"- {k}: {v}")


---

## 3. Exercícios de Fixação e Avaliação

### Questão 1
Explique como o uso da cláusula `needs: [job_anterior]` permite criar sequenciamento e dependência entre jobs no GitHub Actions (garantindo, por exemplo, que a etapa de build só inicie após a aprovação da etapa de testes).

### Questão 2
Como funcionam os **GitHub Actions Secrets**? Escreva o trecho em sintaxe `${{ secrets.NOME_SECRET }}` necessário para injetar uma chave de API armazenada no repositório em uma variável de ambiente do runner.

### Questão 3
Qual a diferença entre a instrução `run:` (que executa comandos shell no runner) e a instrução `uses:` (que invoca uma Action pré-compilada da comunidade) em um *step* do GitHub Actions?
